# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardik144/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The content action queue uses observable content-performance signals to identify pages that may deserve human review. The signals include changes in clicks between the previous and latest 30-day periods, impressions, content age, and available search-performance metrics.

Each row receives a suggested action, a reason code, and a transparent heuristic priority score. This score is a prioritization aid, not a probability of success or a prediction of guaranteed traffic growth.

The actions are grouped into practical categories: refresh declining content, investigate click-through performance, review pages with search visibility but limited clicks, and monitor content with no immediate warning signal.

The queue is intended to support editorial decision-making. Recommendations must be checked against the page, its search intent, business goals, and any recent changes before action is taken.


In [14]:
# ============================================================
# LOAD DATASET FOR WEEK 7
# ============================================================

import pandas as pd
from pathlib import Path

dataset_path = Path("/content/content_refresh_anonymized.csv")

# Try the current working directory if the Colab path is absent.
if not dataset_path.exists():
    dataset_path = Path("content_refresh_anonymized.csv")

if not dataset_path.exists():
    raise FileNotFoundError(
        "Dataset not found. Check the Files panel in Colab "
        "and confirm the location of content_refresh_anonymized.csv."
    )

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [15]:

import pandas as pd
import numpy as np

from IPython.display import display

# ------------------------------------------------------------
# 1. CHECK THE DATASET
# ------------------------------------------------------------

if "df" not in globals():
    raise NameError(
        "The dataset variable 'df' is not defined. "
        "Run the notebook's dataset-loading cell first."
    )

print("Dataset shape:", df.shape)

# Work on a copy. Do not modify the original dataset.
action_df = df.copy()

# ------------------------------------------------------------
# 2. CHECK REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "clicks_last_30d",
    "clicks_prev_30d"
]

missing_columns = [
    col for col in required_columns
    if col not in action_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        "Check the dataset before proceeding."
    )

# Convert performance fields to numeric values.
numeric_columns = [
    "clicks_last_30d",
    "clicks_prev_30d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

for col in numeric_columns:
    if col in action_df.columns:
        action_df[col] = pd.to_numeric(
            action_df[col], errors="coerce"
        )

# ------------------------------------------------------------
# 3. PREPARE PERFORMANCE SIGNALS
# ------------------------------------------------------------

latest_clicks = action_df["clicks_last_30d"].fillna(0).clip(lower=0)
previous_clicks = action_df["clicks_prev_30d"].fillna(0).clip(lower=0)

action_df["click_change"] = latest_clicks - previous_clicks

# Percentage change is calculated only when previous clicks
# are greater than zero. Otherwise, it is left undefined.
action_df["click_change_pct"] = np.where(
    previous_clicks > 0,
    (latest_clicks - previous_clicks) / previous_clicks * 100,
    np.nan
)

# Flag meaningful declines only when there were previous clicks.
action_df["clicks_declined"] = (
    (previous_clicks > 0) &
    (latest_clicks < previous_clicks)
)

# ------------------------------------------------------------
# 4. BUILD REASON CODES AND SUGGESTED ACTIONS
# ------------------------------------------------------------

reason_codes = []
suggested_actions = []
priority_scores = []

# Percentile thresholds are descriptive heuristics calculated
# from this dataset, not trained model decision boundaries.

if "impressions_last_30d" in action_df.columns:
    impressions = action_df["impressions_last_30d"].fillna(0).clip(lower=0)

    positive_impressions = impressions[impressions > 0]

    if len(positive_impressions) > 0:
        high_impression_threshold = positive_impressions.quantile(0.75)
    else:
        high_impression_threshold = np.inf
else:
    impressions = pd.Series(0, index=action_df.index)
    high_impression_threshold = np.inf

if "avg_position" in action_df.columns:
    avg_position = action_df["avg_position"]
else:
    avg_position = pd.Series(np.nan, index=action_df.index)

if "ctr" in action_df.columns:
    ctr = action_df["ctr"]
else:
    ctr = pd.Series(np.nan, index=action_df.index)

if "content_age_days" in action_df.columns:
    content_age = action_df["content_age_days"]
else:
    content_age = pd.Series(np.nan, index=action_df.index)

if "days_since_last_update" in action_df.columns:
    days_since_update = action_df["days_since_last_update"]
else:
    days_since_update = pd.Series(np.nan, index=action_df.index)

# Use the dataset's median age as a descriptive reference.
# This does not mean that older content is automatically stale.
valid_ages = content_age.dropna()

if len(valid_ages) > 0:
    age_reference = valid_ages.median()
else:
    age_reference = np.inf

# CTR may be stored as a fraction or percentage.
# Preserve the dataset's scale and use its median as a reference.
valid_ctr = ctr.dropna()

if len(valid_ctr) > 0:
    ctr_reference = valid_ctr.median()
else:
    ctr_reference = np.nan

for idx in action_df.index:

    reasons = []
    actions = []

    current_clicks = latest_clicks.loc[idx]
    previous = previous_clicks.loc[idx]
    click_change = action_df.loc[idx, "click_change"]

    row_impressions = impressions.loc[idx]
    row_position = avg_position.loc[idx]
    row_ctr = ctr.loc[idx]
    row_age = content_age.loc[idx]
    row_days_since_update = days_since_update.loc[idx]

    # A. Declining clicks
    if previous > 0 and current_clicks < previous:
        reasons.append("CLICK_DECLINE")
        actions.append("Review and consider refreshing the page")

    # B. High impressions and below-median CTR
    if (
        pd.notna(row_ctr)
        and pd.notna(ctr_reference)
        and row_impressions >= high_impression_threshold
        and row_ctr < ctr_reference
    ):
        reasons.append("CTR_REVIEW")
        actions.append("Review title, description, and search intent")

    # C. Search visibility with a relatively weak position
    if (
        row_impressions >= high_impression_threshold
        and pd.notna(row_position)
        and 8 <= row_position <= 20
    ):
        reasons.append("VISIBILITY_OPPORTUNITY")
        actions.append("Review relevance, content depth, and internal links")

    # D. Older content
    if pd.notna(row_age) and row_age > age_reference:
        reasons.append("OLDER_CONTENT_REVIEW")
        actions.append("Check factual accuracy and content freshness")

    # E. Long time since last update
    if pd.notna(row_days_since_update) and row_days_since_update > 90:
        reasons.append("UPDATE_GAP")
        actions.append("Check whether an editorial update is appropriate")

    # F. No warning signals found
    if not reasons:
        reasons.append("MONITOR")
        actions.append("Monitor performance; no immediate action flagged")

    reason_codes.append("; ".join(dict.fromkeys(reasons)))
    suggested_actions.append("; ".join(dict.fromkeys(actions)))

    # --------------------------------------------------------
    # Transparent heuristic priority score
    # --------------------------------------------------------
    # Decline component: larger relative drops receive more weight.
    # Exposure component: pages with more impressions receive
    # more weight, using a percentile rank.
    # Age component: older content can receive a small review weight.
    #
    # This is NOT a model prediction or estimated business value.

    decline_component = 0.0

    if previous > 0 and current_clicks < previous:
        decline_fraction = (previous - current_clicks) / previous
        decline_component = min(max(decline_fraction, 0), 1)

    if len(positive_impressions) > 0:
        exposure_component = (
            impressions.loc[idx] / positive_impressions.max()
        )
        exposure_component = min(max(exposure_component, 0), 1)
    else:
        exposure_component = 0.0

    age_component = 0.0

    if pd.notna(row_age) and pd.notna(age_reference) and age_reference > 0:
        age_component = min(max(row_age / (2 * age_reference), 0), 1)

    # Higher scores indicate higher heuristic review priority.
    priority_score = (
        0.60 * decline_component
        + 0.25 * exposure_component
        + 0.15 * age_component
    )

    priority_scores.append(priority_score)

action_df["reason_codes"] = reason_codes
action_df["suggested_action"] = suggested_actions
action_df["priority_score"] = priority_scores

# ------------------------------------------------------------
# 5. CREATE THE RANKED ACTION QUEUE
# ------------------------------------------------------------

# Include an identifier if one is available.
identifier_columns = [
    col for col in ["content_id", "content_type", "main_intent"]
    if col in action_df.columns
]

queue_columns = identifier_columns + [
    "clicks_prev_30d",
    "clicks_last_30d",
    "click_change",
    "click_change_pct"
]

queue_columns += [
    col for col in [
        "impressions_last_30d",
        "avg_position",
        "ctr",
        "content_age_days",
        "days_since_last_update"
    ]
    if col in action_df.columns
]

queue_columns += [
    "reason_codes",
    "suggested_action",
    "priority_score"
]

# Sort by heuristic priority, then by the absolute click decline.
action_queue = (
    action_df[queue_columns]
    .assign(
        _absolute_click_decline=(
            action_df["click_change"].clip(upper=0).abs()
        )
    )
    .sort_values(
        by=["priority_score", "_absolute_click_decline"],
        ascending=[False, False]
    )
    .drop(columns="_absolute_click_decline")
    .reset_index(drop=True)
)

action_queue.insert(
    0, "review_rank", np.arange(1, len(action_queue) + 1)
)

print("Ranked action queue created.")
print("Rows in queue:", len(action_queue))
print("Reason-code counts:")

display(
    action_queue["reason_codes"]
    .str.split("; ")
    .explode()
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="row_count")
)

print("\nTop 10 candidate actions for human review:")
display(action_queue.head(10))

print(
    "\nInterpretation: rankings are heuristic review priorities "
    "based on observed dataset fields. They do not establish "
    "causality, guarantee improvement, or replace editorial review."
)

Dataset shape: (30000, 44)
Ranked action queue created.
Rows in queue: 30000
Reason-code counts:


,reason_code,row_count
0,OLDER_CONTENT_REVIEW,14354
1,UPDATE_GAP,9345
2,MONITOR,8335
3,CLICK_DECLINE,6806
4,VISIBILITY_OPPORTUNITY,2138
5,CTR_REVIEW,913



Top 10 candidate actions for human review:


,review_rank,content_id,content_type,main_intent,clicks_prev_30d,clicks_last_30d,click_change,click_change_pct,impressions_last_30d,avg_position,ctr,content_age_days,days_since_last_update,reason_codes,suggested_action,priority_score
0,1,content_0d5a75339f4b,keyword article,informational,2,0,-2,-100.0,6943,8.7,0.06,537,25,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.757269
1,2,content_2725d2bcfac1,keyword article,informational,3,0,-3,-100.0,5915,9.1,0.02,504,104,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.756193
2,3,content_bd2aca6e820a,keyword article,transactional,2,0,-2,-100.0,3260,8.1,0.07,480,22,CLICK_DECLINE; VISIBILITY_OPPORTUNITY; OLDER_C...,Review and consider refreshing the page; Revie...,0.753413
3,4,content_93470578e5b4,keyword article,commercial,1,0,-1,-100.0,2582,50.4,0.03,480,22,CLICK_DECLINE; CTR_REVIEW; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Revie...,0.752703
4,5,content_430f0e467f3a,keyword article,informational,5,0,-5,-100.0,2050,6.5,0.10,537,7,CLICK_DECLINE; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Check...,0.752146
5,6,content_ad0db3ac4e45,keyword article,informational,2,0,-2,-100.0,2027,25.9,0.02,557,20,CLICK_DECLINE; CTR_REVIEW; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Revie...,0.752122
6,7,content_554fe5cca7a6,keyword article,commercial,5,0,-5,-100.0,1978,10.2,0.06,480,22,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.752071
7,8,content_6a5b01acb593,keyword article,commercial,1,0,-1,-100.0,1898,14.1,0.04,480,20,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.751987
8,9,content_00603386e331,keyword article,informational,2,0,-2,-100.0,1835,5.9,0.11,487,20,CLICK_DECLINE; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Check...,0.751921
9,10,content_bf54f1b66e17,keyword article,informational,2,0,-2,-100.0,1829,8.2,0.05,545,14,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.751915



Interpretation: rankings are heuristic review priorities based on observed dataset fields. They do not establish causality, guarantee improvement, or replace editorial review.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

#### Intended use

The content action playbook is designed to help SEO analysts and content editors prioritize pages for human review. It uses observed performance signals, such as changes in clicks, impressions, content age, and available search metrics, to suggest possible next actions.

The ranked queue is a decision-support tool. It can help reviewers identify pages that may need refreshing, investigate click-through performance, review search visibility opportunities, or continue monitoring.

#### Who should use it?

* SEO analysts reviewing page performance.
* Content editors deciding which pages may need updates.
* Research and marketing teams investigating changes in organic search performance.

#### What the playbook can do

* Organize pages into a ranked review queue.
* Provide reason codes that explain why a page was flagged.
* Suggest possible editorial actions based on observed signals.
* Help reviewers focus their attention on pages that may warrant investigation.

#### Limits and assumptions

The priority score is a transparent heuristic, not a trained probability model. A higher score does not guarantee that updating a page will increase clicks, impressions, or rankings.

The analysis uses the available dataset and its recorded time windows. Missing values, incomplete historical information, unobserved search-engine changes, seasonality, and external competition may affect interpretation.

The observed relationships are not proof of causation. A decline in clicks does not establish that content quality caused the decline, and older content is not automatically outdated.

The queue should not be treated as a production recommendation system. Its outputs are intended for exploratory research and human-reviewed decision support.

#### When the results may not be valid

The recommendations may be unreliable when the underlying data is incomplete, when search intent has changed, when a page has recently been redesigned, or when major external events affect search demand.

The playbook should also be reviewed before applying it to websites, clients, or time periods that differ substantially from the dataset used in this analysis.

#### Conclusion

This playbook provides a structured way to identify and investigate potential content actions. Its rankings are intended to guide human attention, not replace editorial judgment or guarantee business outcomes.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [17]:

# ============================================================
# SECTION 3: HUMAN REVIEW + THE NO-GO LIST
# ============================================================

import pandas as pd

# 1. Human review rules
review_rules = pd.DataFrame([
    {
        "check": "Verify the evidence",
        "requirement": "Check the page metrics and reason code before accepting an action."
    },
    {
        "check": "Check data freshness",
        "requirement": "Confirm that the underlying data is recent and covers the intended period."
    },
    {
        "check": "Review content quality",
        "requirement": "Check accuracy, relevance, search intent, originality, and readability."
    },
    {
        "check": "Check business context",
        "requirement": "Confirm that the recommendation matches the page purpose and business goals."
    },
    {
        "check": "Approve before publishing",
        "requirement": "A responsible human must approve substantive edits before publication."
    },
    {
        "check": "Record the decision",
        "requirement": "Record whether the action was accepted, modified, or rejected, with a reason."
    }
])

print("=" * 65)
print("HUMAN REVIEW RULES")
print("=" * 65)

display(review_rules)


# 2. No-go list: actions that must not be automated
no_go_list = pd.DataFrame([
    {
        "prohibited_action": "Automatic publishing",
        "reason": "Content changes require editorial and factual review."
    },
    {
        "prohibited_action": "Deleting or redirecting pages automatically",
        "reason": "These decisions can affect traffic, links, and business value."
    },
    {
        "prohibited_action": "Making unsupported factual or medical claims",
        "reason": "Claims must be verified by an appropriate human reviewer."
    },
    {
        "prohibited_action": "Changing prices, legal terms, or contractual content",
        "reason": "These changes require authorized business or legal approval."
    },
    {
        "prohibited_action": "Treating model predictions as guaranteed outcomes",
        "reason": "Predictions are uncertain and do not establish causation."
    },
    {
        "prohibited_action": "Using private or unauthorized personal data",
        "reason": "Recommendations must respect privacy and data permissions."
    },
    {
        "prohibited_action": "Making high-impact changes without approval",
        "reason": "Significant changes require human judgment and accountability."
    }
])

print("\n" + "=" * 65)
print("NO-GO LIST")
print("=" * 65)

display(no_go_list)


# 3. Final review policy
print("\nHUMAN REVIEW POLICY")
print("-" * 65)

print(
    "The action playbook provides decision support only. "
    "It does not publish, delete, or modify content automatically."
)

print(
    "Every proposed action must be checked for evidence, "
    "data freshness, content quality, and business relevance."
)

print(
    "A human reviewer is responsible for the final decision. "
    "Rejected or modified recommendations should include a reason."
)

print(
    "\nSection 3 completed: human review requirements and "
    "no-go cases documented."
)

HUMAN REVIEW RULES


,check,requirement
0,Verify the evidence,Check the page metrics and reason code before ...
1,Check data freshness,Confirm that the underlying data is recent and...
2,Review content quality,"Check accuracy, relevance, search intent, orig..."
3,Check business context,Confirm that the recommendation matches the pa...
4,Approve before publishing,A responsible human must approve substantive e...
5,Record the decision,"Record whether the action was accepted, modifi..."



NO-GO LIST


,prohibited_action,reason
0,Automatic publishing,Content changes require editorial and factual ...
1,Deleting or redirecting pages automatically,"These decisions can affect traffic, links, and..."
2,Making unsupported factual or medical claims,Claims must be verified by an appropriate huma...
3,"Changing prices, legal terms, or contractual c...",These changes require authorized business or l...
4,Treating model predictions as guaranteed outcomes,Predictions are uncertain and do not establish...
5,Using private or unauthorized personal data,Recommendations must respect privacy and data ...
6,Making high-impact changes without approval,Significant changes require human judgment and...



HUMAN REVIEW POLICY
-----------------------------------------------------------------
The action playbook provides decision support only. It does not publish, delete, or modify content automatically.
Every proposed action must be checked for evidence, data freshness, content quality, and business relevance.
A human reviewer is responsible for the final decision. Rejected or modified recommendations should include a reason.

Section 3 completed: human review requirements and no-go cases documented.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [18]:

# SECTION 4: MONITORING / RETRAIN TRIGGERS

import pandas as pd

monitoring_plan = pd.DataFrame([
    {
        "trigger": "Prediction error increases",
        "check": "Evaluate MAE against the baseline on newly available labeled data.",
        "threshold": "Model MAE is more than 20% above its validation MAE.",
        "action": "Investigate the error and consider retraining after review."
    },
    {
        "trigger": "Feature distribution drift",
        "check": "Compare current feature distributions with the training data.",
        "threshold": "A major feature distribution change is detected.",
        "action": "Investigate changes in traffic, content, or data collection."
    },
    {
        "trigger": "Content performance changes",
        "check": "Compare recent clicks and impressions with previous periods.",
        "threshold": "A sustained change in performance is observed.",
        "action": "Review whether existing recommendations remain relevant."
    },
    {
        "trigger": "Data quality problems",
        "check": "Check missing values, invalid values, and unexpected categories.",
        "threshold": "Required features or target values are missing or invalid.",
        "action": "Fix the data issue before using predictions."
    },
    {
        "trigger": "Scheduled review",
        "check": "Review model metrics and recommendations monthly.",
        "threshold": "Monthly review date is reached.",
        "action": "Inspect performance and decide whether retraining is needed."
    }
])

print("MONITORING AND RETRAIN TRIGGERS")
display(monitoring_plan)

print(
    "\nRetraining policy: Retraining is considered only after a trigger "
    "is investigated, sufficient new labeled data is available, and "
    "the updated model passes validation against the baseline."
)

print(
    "\nImportant: These thresholds are proposed monitoring rules, "
    "not experimentally established guarantees. All retraining "
    "and recommendation changes require human approval."
)

MONITORING AND RETRAIN TRIGGERS


,trigger,check,threshold,action
0,Prediction error increases,Evaluate MAE against the baseline on newly ava...,Model MAE is more than 20% above its validatio...,Investigate the error and consider retraining ...
1,Feature distribution drift,Compare current feature distributions with the...,A major feature distribution change is detected.,"Investigate changes in traffic, content, or da..."
2,Content performance changes,Compare recent clicks and impressions with pre...,A sustained change in performance is observed.,Review whether existing recommendations remain...
3,Data quality problems,"Check missing values, invalid values, and unex...",Required features or target values are missing...,Fix the data issue before using predictions.
4,Scheduled review,Review model metrics and recommendations monthly.,Monthly review date is reached.,Inspect performance and decide whether retrain...



Retraining policy: Retraining is considered only after a trigger is investigated, sufficient new labeled data is available, and the updated model passes validation against the baseline.

Important: These thresholds are proposed monitoring rules, not experimentally established guarantees. All retraining and recommendation changes require human approval.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [19]:

# SECTION 5: EXPORTS FOR THE PAPER

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# Create output directories
output_dir = Path("work/outputs")
figures_dir = Path("work/figures")

output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

# 1. Export the ranked action queue
if "action_queue" not in globals():
    raise NameError(
        "action_queue is not defined. Run Section 1 first."
    )

if not isinstance(action_queue, pd.DataFrame):
    raise TypeError("action_queue must be a pandas DataFrame.")

if action_queue.empty:
    raise ValueError("The action queue is empty. Check Section 1.")

queue_path = output_dir / "w07_ranked_action_queue.csv"

action_queue.to_csv(queue_path, index=False)

print("Ranked action queue exported:", queue_path)
print("Number of actions:", len(action_queue))

# 2. Save existing figures for the research paper
figure_numbers = plt.get_fignums()
saved_figures = []

for i, figure_number in enumerate(figure_numbers, start=1):
    fig = plt.figure(figure_number)

    figure_path = figures_dir / f"w07_figure_{i}.png"

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    saved_figures.append(str(figure_path))

print("\nFigures saved:", len(saved_figures))

for figure_path in saved_figures:
    print(figure_path)

# 3. Export an artifact receipt
export_summary = {
    "assignment": "ML-10 Content Action Playbook",
    "queue_file": str(queue_path),
    "number_of_actions": int(len(action_queue)),
    "queue_columns": action_queue.columns.tolist(),
    "figures_saved": saved_figures,
    "figure_count": len(saved_figures)
}

summary_path = output_dir / "w07_export_summary.json"

with open(summary_path, "w") as f:
    json.dump(export_summary, f, indent=2)

print("\nExport summary saved:", summary_path)

# 4. Display a preview
print("\nPREVIEW OF EXPORTED ACTION QUEUE")
display(action_queue.head(10))

print("\nSECTION 5 EXPORTS COMPLETED")

Ranked action queue exported: work/outputs/w07_ranked_action_queue.csv
Number of actions: 30000

Figures saved: 0

Export summary saved: work/outputs/w07_export_summary.json

PREVIEW OF EXPORTED ACTION QUEUE


,review_rank,content_id,content_type,main_intent,clicks_prev_30d,clicks_last_30d,click_change,click_change_pct,impressions_last_30d,avg_position,ctr,content_age_days,days_since_last_update,reason_codes,suggested_action,priority_score
0,1,content_0d5a75339f4b,keyword article,informational,2,0,-2,-100.0,6943,8.7,0.06,537,25,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.757269
1,2,content_2725d2bcfac1,keyword article,informational,3,0,-3,-100.0,5915,9.1,0.02,504,104,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.756193
2,3,content_bd2aca6e820a,keyword article,transactional,2,0,-2,-100.0,3260,8.1,0.07,480,22,CLICK_DECLINE; VISIBILITY_OPPORTUNITY; OLDER_C...,Review and consider refreshing the page; Revie...,0.753413
3,4,content_93470578e5b4,keyword article,commercial,1,0,-1,-100.0,2582,50.4,0.03,480,22,CLICK_DECLINE; CTR_REVIEW; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Revie...,0.752703
4,5,content_430f0e467f3a,keyword article,informational,5,0,-5,-100.0,2050,6.5,0.10,537,7,CLICK_DECLINE; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Check...,0.752146
5,6,content_ad0db3ac4e45,keyword article,informational,2,0,-2,-100.0,2027,25.9,0.02,557,20,CLICK_DECLINE; CTR_REVIEW; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Revie...,0.752122
6,7,content_554fe5cca7a6,keyword article,commercial,5,0,-5,-100.0,1978,10.2,0.06,480,22,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.752071
7,8,content_6a5b01acb593,keyword article,commercial,1,0,-1,-100.0,1898,14.1,0.04,480,20,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.751987
8,9,content_00603386e331,keyword article,informational,2,0,-2,-100.0,1835,5.9,0.11,487,20,CLICK_DECLINE; OLDER_CONTENT_REVIEW,Review and consider refreshing the page; Check...,0.751921
9,10,content_bf54f1b66e17,keyword article,informational,2,0,-2,-100.0,1829,8.2,0.05,545,14,CLICK_DECLINE; CTR_REVIEW; VISIBILITY_OPPORTUN...,Review and consider refreshing the page; Revie...,0.751915



SECTION 5 EXPORTS COMPLETED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.